In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
!nvidia-smi

Fri Sep 11 09:40:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if files:
        print(root)
        print(files[:5])

/kaggle/input/datasets/gillesokhin/nutrition5k-dataset
['dish_nutrition_values.csv', 'ingredients_metadata.csv', 'dish_ingredients.csv']
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1564588859
['depth_raw.png', 'rgb.png', 'depth_color.png']
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1561480439
['depth_raw.png', 'rgb.png', 'depth_color.png']
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1562615388
['depth_raw.png', 'rgb.png', 'depth_color.png']
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1560961059
['depth_raw.png', 'rgb.png', 'depth_color.png']
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1558459020
['depth_raw.png', 'rgb.png', 'depth_color.png']
/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead/dish_1562703408
['depth_raw.png', 'rgb.png', 'depth

KeyboardInterrupt: 

In [4]:
import pandas as pd

BASE = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein']


,dish_id,calories,mass,fat,carb,protein
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000


In [5]:
import os

nutrition["image_path"] = nutrition["dish_id"].apply(
    lambda x: f"{BASE}/imagery/realsense_overhead/{x}/rgb.png"
)

# chỉ giữ món thực sự có ảnh RGB
nutrition = nutrition[
    nutrition["image_path"].apply(os.path.exists)
].reset_index(drop=True)

print("Số món có ảnh:", len(nutrition))

display(
    nutrition[
        ["dish_id", "image_path", "total_calories"]
    ].head()
)

Số món có ảnh: 3244


KeyError: "['total_calories'] not in index"

In [6]:
print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein', 'image_path']


,dish_id,calories,mass,fat,carb,protein,image_path
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...
1,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...
2,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...
3,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...
4,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...


In [7]:
df = nutrition[
    ["dish_id", "image_path", "calories"]
].copy()

display(df.head())
print("Tổng số mẫu:", len(df))

,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


Tổng số mẫu: 3244


In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2270
Val: 487
Test: 487


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(row["calories"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

In [12]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [13]:
train_dataset = NutritionDataset(train_df, train_transform)
val_dataset = NutritionDataset(val_df, val_transform)
test_dataset = NutritionDataset(test_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

2270 487 487


In [14]:
images, calories = next(iter(train_loader))

print("Images:", images.shape)
print("Calories:", calories.shape)
print("Ví dụ calories:", calories[:5])

Images: torch.Size([64, 3, 224, 224])
Calories: torch.Size([64])
Ví dụ calories: tensor([140.5629, 300.7789,  70.2000, 121.7423,  55.8900])


In [15]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Thay lớp cuối classification bằng regression output 1 giá trị calories
model.fc = nn.Linear(model.fc.in_features, 1)

model = model.to(device)

print(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 152MB/s] 


cuda:0


In [16]:
criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    for images, calories in loader:
        images = images.to(device)
        calories = calories.to(device)

        optimizer.zero_grad()

        outputs = model(images).squeeze(1)

        loss = criterion(outputs, calories)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

In [18]:
import numpy as np

def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    preds = []
    targets = []

    with torch.no_grad():
        for images, calories in loader:
            images = images.to(device)
            calories = calories.to(device)

            outputs = model(images).squeeze(1)

            loss = criterion(outputs, calories)

            total_loss += loss.item() * images.size(0)

            preds.extend(outputs.cpu().numpy())
            targets.extend(calories.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(np.mean((preds - targets) ** 2))

    return total_loss / len(loader.dataset), mae, rmse

In [36]:
%%writefile /kaggle/working/train_ddp_fair.py

import os
import time
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights


# ============================================================
# CONFIG
# ============================================================
SEED = 42
EPOCHS = 20
LOCAL_BATCH_SIZE = 32      # 32 x 2 GPU = global batch 64
LR = 1e-4


class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(
            row["calories"],
            dtype=torch.float32
        )

        if self.transform:
            image = self.transform(image)

        return image, calories


def evaluate(model, loader, device):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for images, calories in loader:
            images = images.to(device, non_blocking=True)
            calories = calories.to(device, non_blocking=True)

            outputs = model(images).squeeze(1)

            preds.extend(outputs.cpu().numpy())
            targets.extend(calories.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(np.mean((preds - targets) ** 2))

    return mae, rmse


def main():

    # ========================================================
    # DDP SETUP
    # ========================================================
    dist.init_process_group(backend="nccl")

    rank = dist.get_rank()
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = dist.get_world_size()

    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    # Seed
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    # ========================================================
    # DATA
    # ========================================================
    train_df = pd.read_csv("/kaggle/working/train.csv")
    val_df = pd.read_csv("/kaggle/working/val.csv")

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dataset = NutritionDataset(
        train_df,
        train_transform
    )

    val_dataset = NutritionDataset(
        val_df,
        val_transform
    )

    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=SEED
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=LOCAL_BATCH_SIZE,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True
    )

    # Validation chỉ rank 0 dùng
    if rank == 0:
        val_loader = DataLoader(
            val_dataset,
            batch_size=64,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

    # ========================================================
    # MODEL
    # ========================================================
    model = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        1
    )

    model = model.to(device)

    model = DDP(
        model,
        device_ids=[local_rank]
    )

    criterion = nn.SmoothL1Loss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    # ========================================================
    # TRAIN
    # ========================================================
    total_train_time = 0.0
    best_mae = float("inf")
    final_mae = None
    final_rmse = None

    history = []

    for epoch in range(EPOCHS):

        train_sampler.set_epoch(epoch)
        model.train()

        # Đồng bộ trước khi đo
        dist.barrier()
        torch.cuda.synchronize()

        start_time = time.time()

        total_loss = 0.0
        total_samples = 0

        for images, calories in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            calories = calories.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images).squeeze(1)

            loss = criterion(
                outputs,
                calories
            )

            loss.backward()
            optimizer.step()

            total_loss += (
                loss.item() * images.size(0)
            )

            total_samples += images.size(0)

        torch.cuda.synchronize()
        dist.barrier()

        # CHỈ thời gian training
        epoch_train_time = (
            time.time() - start_time
        )

        total_train_time += epoch_train_time

        # ================================================
        # VALIDATION - KHÔNG TÍNH VÀO TRAIN TIME
        # ================================================
        if rank == 0:

            val_mae, val_rmse = evaluate(
                model.module,
                val_loader,
                device
            )

            final_mae = val_mae
            final_rmse = val_rmse

            if val_mae < best_mae:

                best_mae = val_mae

                torch.save(
                    model.module.state_dict(),
                    "/kaggle/working/"
                    "resnet18_20epoch_2gpu_ddp_best.pth"
                )

            history.append({
                "epoch": epoch + 1,
                "val_mae": val_mae,
                "val_rmse": val_rmse,
                "train_time_sec": epoch_train_time
            })

            print(
                f"Epoch {epoch+1:02d}/{EPOCHS} | "
                f"MAE: {val_mae:.2f} kcal | "
                f"RMSE: {val_rmse:.2f} kcal | "
                f"Train: {epoch_train_time:.2f}s"
            )

        # rank 1 đợi rank 0 validate
        dist.barrier()

    # ========================================================
    # RESULT
    # ========================================================
    if rank == 0:

        throughput = (
            len(train_dataset) * EPOCHS
            / total_train_time
        )

        history_df = pd.DataFrame(history)

        history_df.to_csv(
            "/kaggle/working/"
            "resnet18_20epoch_2gpu_history.csv",
            index=False
        )

        print(
            "\n=== 2 GPU DDP - 20 EPOCH RESULT ==="
        )

        print(
            f"Training time: "
            f"{total_train_time:.2f} s"
        )

        print(
            f"Throughput: "
            f"{throughput:.2f} images/s"
        )

        print(
            f"Best MAE: "
            f"{best_mae:.2f} kcal"
        )

        print(
            f"Final MAE: "
            f"{final_mae:.2f} kcal"
        )

        print(
            f"Final RMSE: "
            f"{final_rmse:.2f} kcal"
        )

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_ddp_fair.py


In [37]:
!torchrun \
    --standalone \
    --nproc_per_node=2 \
    /kaggle/working/train_ddp_fair.py

W0911 10:48:48.817000 1533 torch/distributed/run.py:852] 
W0911 10:48:48.817000 1533 torch/distributed/run.py:852] *****************************************
W0911 10:48:48.817000 1533 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0911 10:48:48.817000 1533 torch/distributed/run.py:852] *****************************************
[W911 10:48:49.769264149 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W911 10:48:52.273321908 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W911 10:48:52.381578037 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You ca

In [47]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision.models import resnet50, ResNet50_Weights


# ============================================================
# CONFIG
# ============================================================
SEED = 42
EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0")


# ============================================================
# DATALOADER
# giữ nguyên dataset/split từ experiment trước
# ============================================================
train_loader_r50 = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_r50 = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# MODEL
# ============================================================
model_r50 = resnet50(
    weights=ResNet50_Weights.DEFAULT
)

model_r50.fc = nn.Linear(
    model_r50.fc.in_features,
    1
)

model_r50 = model_r50.to(device)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model_r50.parameters(),
    lr=LR
)


# ============================================================
# TRAIN FUNCTION
# ============================================================
def train_one_epoch_r50(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0

    for images, calories in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        calories = calories.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(images).squeeze(1)

        loss = criterion(
            outputs,
            calories
        )

        loss.backward()
        optimizer.step()

        total_loss += (
            loss.item() * images.size(0)
        )

    return total_loss / len(loader.dataset)


# ============================================================
# TRAIN 20 EPOCH
# ============================================================
history_r50 = []

best_mae = float("inf")
total_train_time = 0.0

for epoch in range(EPOCHS):

    torch.cuda.synchronize()
    start = time.time()

    train_loss = train_one_epoch_r50(
        model_r50,
        train_loader_r50,
        criterion,
        optimizer,
        device
    )

    torch.cuda.synchronize()

    epoch_train_time = (
        time.time() - start
    )

    total_train_time += epoch_train_time

    # Validation KHÔNG tính vào training time
    val_loss, val_mae, val_rmse = evaluate(
        model_r50,
        val_loader_r50,
        criterion,
        device
    )

    history_r50.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "train_time_sec": epoch_train_time
    })

    # Save best model
    if val_mae < best_mae:

        best_mae = val_mae

        torch.save(
            model_r50.state_dict(),
            "/kaggle/working/"
            "resnet50_20epoch_1gpu_best.pth"
        )

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.2f} | "
        f"MAE: {val_mae:.2f} kcal | "
        f"RMSE: {val_rmse:.2f} kcal | "
        f"Train: {epoch_train_time:.2f}s"
    )


# ============================================================
# SAVE HISTORY
# ============================================================
history_r50_df = pd.DataFrame(
    history_r50
)

history_r50_df.to_csv(
    "/kaggle/working/"
    "resnet50_20epoch_1gpu_history.csv",
    index=False
)


# ============================================================
# RESULT
# ============================================================
throughput = (
    len(train_dataset) * EPOCHS
    / total_train_time
)

print(
    "\n=== ResNet50 - 1 GPU - 20 EPOCH RESULT ==="
)

print(
    f"Training time: "
    f"{total_train_time:.2f} s"
)

print(
    f"Throughput: "
    f"{throughput:.2f} images/s"
)

print(
    f"Best MAE: "
    f"{best_mae:.2f} kcal"
)

print(
    f"Final MAE: "
    f"{history_r50[-1]['val_mae']:.2f} kcal"
)

print(
    f"Final RMSE: "
    f"{history_r50[-1]['val_rmse']:.2f} kcal"
)

Epoch 01/20 | Train Loss: 247.78 | MAE: 194.26 kcal | RMSE: 270.18 kcal | Train: 28.84s
Epoch 02/20 | Train Loss: 229.30 | MAE: 206.70 kcal | RMSE: 280.59 kcal | Train: 27.44s
Epoch 03/20 | Train Loss: 212.94 | MAE: 223.20 kcal | RMSE: 296.85 kcal | Train: 27.29s
Epoch 04/20 | Train Loss: 201.21 | MAE: 206.75 kcal | RMSE: 277.52 kcal | Train: 27.88s
Epoch 05/20 | Train Loss: 191.07 | MAE: 183.99 kcal | RMSE: 253.01 kcal | Train: 27.43s
Epoch 06/20 | Train Loss: 181.71 | MAE: 175.25 kcal | RMSE: 243.76 kcal | Train: 27.25s
Epoch 07/20 | Train Loss: 173.08 | MAE: 179.47 kcal | RMSE: 244.71 kcal | Train: 27.80s
Epoch 08/20 | Train Loss: 164.46 | MAE: 140.71 kcal | RMSE: 200.79 kcal | Train: 27.58s
Epoch 09/20 | Train Loss: 155.67 | MAE: 129.06 kcal | RMSE: 186.68 kcal | Train: 27.27s
Epoch 10/20 | Train Loss: 146.77 | MAE: 144.62 kcal | RMSE: 205.76 kcal | Train: 27.35s
Epoch 11/20 | Train Loss: 138.94 | MAE: 139.78 kcal | RMSE: 196.25 kcal | Train: 27.46s
Epoch 12/20 | Train Loss: 129.94

In [48]:
%%writefile /kaggle/working/train_resnet50_ddp_fair.py

import os
import time
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights


# ============================================================
# CONFIG - GIỮ GIỐNG 1 GPU
# ============================================================
SEED = 42
EPOCHS = 20
LOCAL_BATCH_SIZE = 32       # 32 x 2 GPU = global batch 64
LR = 1e-4


class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        calories = torch.tensor(
            row["calories"],
            dtype=torch.float32
        )

        if self.transform:
            image = self.transform(image)

        return image, calories


def evaluate(model, loader, device):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for images, calories in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            calories = calories.to(
                device,
                non_blocking=True
            )

            outputs = model(images).squeeze(1)

            preds.extend(
                outputs.cpu().numpy()
            )

            targets.extend(
                calories.cpu().numpy()
            )

    preds = np.array(preds)
    targets = np.array(targets)

    mae = np.mean(
        np.abs(preds - targets)
    )

    rmse = np.sqrt(
        np.mean(
            (preds - targets) ** 2
        )
    )

    return mae, rmse


def main():

    # ========================================================
    # DDP INIT
    # ========================================================
    dist.init_process_group(
        backend="nccl"
    )

    rank = dist.get_rank()
    local_rank = int(
        os.environ["LOCAL_RANK"]
    )

    world_size = dist.get_world_size()

    torch.cuda.set_device(local_rank)

    device = torch.device(
        f"cuda:{local_rank}"
    )

    # Seed
    random.seed(SEED)
    np.random.seed(SEED)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


    # ========================================================
    # DATA
    # ========================================================
    train_df = pd.read_csv(
        "/kaggle/working/train.csv"
    )

    val_df = pd.read_csv(
        "/kaggle/working/val.csv"
    )

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dataset = NutritionDataset(
        train_df,
        train_transform
    )

    val_dataset = NutritionDataset(
        val_df,
        val_transform
    )


    # ========================================================
    # DISTRIBUTED SAMPLER
    # ========================================================
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=SEED
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=LOCAL_BATCH_SIZE,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True
    )

    # Validation chỉ rank 0
    if rank == 0:

        val_loader = DataLoader(
            val_dataset,
            batch_size=64,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )


    # ========================================================
    # RESNET50
    # ========================================================
    model = resnet50(
        weights=ResNet50_Weights.DEFAULT
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        1
    )

    model = model.to(device)

    model = DDP(
        model,
        device_ids=[local_rank]
    )

    criterion = nn.SmoothL1Loss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )


    # ========================================================
    # TRAIN
    # ========================================================
    total_train_time = 0.0
    best_mae = float("inf")

    final_mae = None
    final_rmse = None

    history = []

    for epoch in range(EPOCHS):

        train_sampler.set_epoch(epoch)

        model.train()

        # ----------------------------
        # START TRAIN TIMER
        # ----------------------------
        dist.barrier()
        torch.cuda.synchronize()

        start = time.time()

        total_loss = 0.0
        total_samples = 0

        for images, calories in train_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            calories = calories.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(
                images
            ).squeeze(1)

            loss = criterion(
                outputs,
                calories
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item()
                * images.size(0)
            )

            total_samples += (
                images.size(0)
            )

        torch.cuda.synchronize()
        dist.barrier()

        epoch_train_time = (
            time.time() - start
        )

        total_train_time += (
            epoch_train_time
        )

        # ----------------------------
        # VALIDATION
        # KHÔNG TÍNH VÀO TRAIN TIME
        # ----------------------------
        if rank == 0:

            val_mae, val_rmse = evaluate(
                model.module,
                val_loader,
                device
            )

            final_mae = val_mae
            final_rmse = val_rmse

            if val_mae < best_mae:

                best_mae = val_mae

                torch.save(
                    model.module.state_dict(),
                    "/kaggle/working/"
                    "resnet50_20epoch_2gpu_ddp_best.pth"
                )

            history.append({
                "epoch": epoch + 1,
                "val_mae": val_mae,
                "val_rmse": val_rmse,
                "train_time_sec": epoch_train_time
            })

            print(
                f"Epoch {epoch+1:02d}/{EPOCHS} | "
                f"MAE: {val_mae:.2f} kcal | "
                f"RMSE: {val_rmse:.2f} kcal | "
                f"Train: {epoch_train_time:.2f}s"
            )

        # rank 1 đợi validation xong
        dist.barrier()


    # ========================================================
    # FINAL RESULT
    # ========================================================
    if rank == 0:

        throughput = (
            len(train_dataset)
            * EPOCHS
            / total_train_time
        )

        history_df = pd.DataFrame(
            history
        )

        history_df.to_csv(
            "/kaggle/working/"
            "resnet50_20epoch_2gpu_history.csv",
            index=False
        )

        print(
            "\n=== ResNet50 - 2 GPU DDP - 20 EPOCH RESULT ==="
        )

        print(
            f"Training time: "
            f"{total_train_time:.2f} s"
        )

        print(
            f"Throughput: "
            f"{throughput:.2f} images/s"
        )

        print(
            f"Best MAE: "
            f"{best_mae:.2f} kcal"
        )

        print(
            f"Final MAE: "
            f"{final_mae:.2f} kcal"
        )

        print(
            f"Final RMSE: "
            f"{final_rmse:.2f} kcal"
        )

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_resnet50_ddp_fair.py


In [49]:
!torchrun \
    --standalone \
    --nproc_per_node=2 \
    /kaggle/working/train_resnet50_ddp_fair.py

W0911 11:26:07.884000 2691 torch/distributed/run.py:852] 
W0911 11:26:07.884000 2691 torch/distributed/run.py:852] *****************************************
W0911 11:26:07.884000 2691 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0911 11:26:07.884000 2691 torch/distributed/run.py:852] *****************************************
[W911 11:26:08.793833724 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W911 11:26:11.206874601 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W911 11:26:11.245215016 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You ca

In [50]:
import pandas as pd

summary_r50 = pd.DataFrame([
    {
        "experiment": "ResNet50_20epoch_1GPU",
        "model": "ResNet50",
        "gpu_count": 1,
        "epochs": 20,
        "global_batch_size": 64,
        "training_time_sec": 550.37,
        "throughput_img_sec": 82.49,
        "best_mae_kcal": 77.41,
        "final_mae_kcal": 77.41,
        "final_rmse_kcal": 119.34,
        "speedup": 1.00,
        "parallel_efficiency_percent": 100.0
    },
    {
        "experiment": "ResNet50_20epoch_2GPU_DDP",
        "model": "ResNet50",
        "gpu_count": 2,
        "epochs": 20,
        "global_batch_size": 64,
        "training_time_sec": 390.44,
        "throughput_img_sec": 116.28,
        "best_mae_kcal": 84.53,
        "final_mae_kcal": 118.42,
        "final_rmse_kcal": 165.92,
        "speedup": 550.37 / 390.44,
        "parallel_efficiency_percent": (550.37 / 390.44) / 2 * 100
    }
])

summary_r50.to_csv(
    "/kaggle/working/resnet50_20epoch_comparison.csv",
    index=False
)

display(summary_r50)

,experiment,model,gpu_count,epochs,global_batch_size,training_time_sec,throughput_img_sec,best_mae_kcal,final_mae_kcal,final_rmse_kcal,speedup,parallel_efficiency_percent
0,ResNet50_20epoch_1GPU,ResNet50,1,20,64,550.37,82.49,77.41,77.41,119.34,1.000000,100.00000
1,ResNet50_20epoch_2GPU_DDP,ResNet50,2,20,64,390.44,116.28,84.53,118.42,165.92,1.409615,70.48074


In [51]:
!mkdir -p /kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/models
!mkdir -p /kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits
!mkdir -p /kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results
!mkdir -p /kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/scripts

In [52]:
# Models
!cp /kaggle/working/resnet50_20epoch_1gpu_best.pth \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/models/

!cp /kaggle/working/resnet50_20epoch_2gpu_ddp_best.pth \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/models/

# Dataset splits
!cp /kaggle/working/train.csv \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits/

!cp /kaggle/working/val.csv \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits/

!cp /kaggle/working/test.csv \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits/

# Results
!cp /kaggle/working/resnet50_20epoch_1gpu_history.csv \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results/

!cp /kaggle/working/resnet50_20epoch_2gpu_history.csv \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results/

!cp /kaggle/working/resnet50_20epoch_comparison.csv \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results/

# Script
!cp /kaggle/working/train_resnet50_ddp_fair.py \
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/scripts/

In [53]:
!find /kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare -type f

/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/models/resnet50_20epoch_2gpu_ddp_best.pth
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/models/resnet50_20epoch_1gpu_best.pth
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/scripts/train_resnet50_ddp_fair.py
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results/resnet50_20epoch_2gpu_history.csv
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results/resnet50_20epoch_comparison.csv
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/results/resnet50_20epoch_1gpu_history.csv
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits/test.csv
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits/val.csv
/kaggle/working/archive/exp_003_resnet50_20epoch_fair_compare/splits/train.csv


In [54]:
!cd /kaggle/working/archive && \
zip -r /kaggle/working/exp_003_resnet50_20epoch_fair_compare.zip \
exp_003_resnet50_20epoch_fair_compare

  adding: exp_003_resnet50_20epoch_fair_compare/ (stored 0%)
  adding: exp_003_resnet50_20epoch_fair_compare/models/ (stored 0%)
  adding: exp_003_resnet50_20epoch_fair_compare/models/resnet50_20epoch_2gpu_ddp_best.pth (deflated 7%)
  adding: exp_003_resnet50_20epoch_fair_compare/models/resnet50_20epoch_1gpu_best.pth (deflated 7%)
  adding: exp_003_resnet50_20epoch_fair_compare/scripts/ (stored 0%)
  adding: exp_003_resnet50_20epoch_fair_compare/scripts/train_resnet50_ddp_fair.py (deflated 73%)
  adding: exp_003_resnet50_20epoch_fair_compare/results/ (stored 0%)
  adding: exp_003_resnet50_20epoch_fair_compare/results/resnet50_20epoch_2gpu_history.csv (deflated 46%)
  adding: exp_003_resnet50_20epoch_fair_compare/results/resnet50_20epoch_comparison.csv (deflated 35%)
  adding: exp_003_resnet50_20epoch_fair_compare/results/resnet50_20epoch_1gpu_history.csv (deflated 47%)
  adding: exp_003_resnet50_20epoch_fair_compare/splits/ (stored 0%)
  adding: exp_003_resnet50_20epoch_fair_compare/sp

In [55]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

display(
    FileLink(
        "exp_003_resnet50_20epoch_fair_compare.zip"
    )
)

/kaggle/working/exp_003_resnet50_20epoch_fair_compare.zip